# F4-multivar-calculus — Practice p17 — Solution

Let $z_n=\sum_ku_kX_{n,k}$ and $r_n=y_n-cz_n-b$; since $\partial r_n/\partial u_k=-cX_{n,k}$, $\partial M/\partial u_k=-(2c/N)\sum_nr_nX_{n,k}$. The remaining derivatives are $\partial M/\partial c=-(2/N)\sum_nr_nz_n$ and $\partial M/\partial b=-(2/N)\sum_nr_n$.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (50, 3))
y = rng.normal(0, 1, 50)

def m_gradients(u, c, b):
    z = (X * u).sum(axis=1)
    residual = y - c * z - b
    grad_u = (-2.0 * c / X.shape[0]) * (residual[:, None] * X).sum(axis=0)
    grad_c = (-2.0 / X.shape[0]) * (residual * z).sum()
    grad_b = (-2.0 / X.shape[0]) * residual.sum()
    return grad_u, grad_c, grad_b

def m_value(candidate_u, candidate_c, candidate_b):
    z = (X * candidate_u).sum(axis=1)
    return ((y - candidate_c * z - candidate_b)**2).mean()

u = np.array([0.4, -0.2, 0.7]); c = 1.5; b = -0.3
grad_u, grad_c, grad_b = m_gradients(u, c, b)
h = 1e-6
numeric_u = np.array([
    (m_value(u + np.array([h, 0.0, 0.0]), c, b) - m_value(u - np.array([h, 0.0, 0.0]), c, b)) / (2*h),
    (m_value(u + np.array([0.0, h, 0.0]), c, b) - m_value(u - np.array([0.0, h, 0.0]), c, b)) / (2*h),
    (m_value(u + np.array([0.0, 0.0, h]), c, b) - m_value(u - np.array([0.0, 0.0, h]), c, b)) / (2*h)
])
numeric_c = (m_value(u, c + h, b) - m_value(u, c - h, b)) / (2*h)
numeric_b = (m_value(u, c, b + h) - m_value(u, c, b - h)) / (2*h)
max_check_gap = np.max(np.abs(np.concatenate((grad_u - numeric_u, np.array([grad_c - numeric_c, grad_b - numeric_b])))))

The factor $c$ is the derivative of the outer prediction $cz_n+b$ with respect to the intermediate value $z_n$. It scales how a change in $u_k$ propagates through $z_n$ to the prediction.

### Answer check

In [ ]:
assert np.allclose(grad_u, numeric_u, rtol=0.0, atol=1e-8)
assert np.isclose(grad_c, numeric_c, rtol=0.0, atol=1e-8)
assert np.isclose(grad_b, numeric_b, rtol=0.0, atol=1e-8)
assert max_check_gap < 1e-8